In [ ]:
import pandas as pd 
import numpy as np 
import scanpy as sc
import pycisTopic
pycisTopic.__version__

import os
out_dir = "scenic_noqc_outs"
os.makedirs(out_dir, exist_ok = True)

## 概述

在这一版中，尝试取消或者降低QC标准，保留更多细胞，同时分细胞类型保存cistopic object，之后再做LDA推断

## get cell anno 

In [ ]:
"""
obs_list = []
path_list = ['/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/cy_atac.h5ad',
             '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/qf_atac.h5ad',
             '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/jt_atac.h5ad',
             '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/m_atac.h5ad',
             '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/t_atac.h5ad'
]

for path in path_list:
    scdata = sc.read_h5ad(path)
    obs_list.append(scdata.obs.copy())

obs =  pd.concat(obs_list)

sample = []
for ele in obs.loc[:,'species'].values:
    sample.append(f'{ele}HM')
obs.loc[:,'sample'] = sample

new_idx = obs.index.values + '-' + np.array(sample)
obs.index = new_idx

obs.to_csv(os.path.join(out_dir,'cell_data.csv'), index = True)

cell_data = obs
"""

In [ ]:
cell_data = pd.read_csv(os.path.join(out_dir,'cell_data.csv'), index_col = 0)
cell_data.head()

## get pseudobulk profiles from cell annotations

In [ ]:
chromsizes = pd.read_table(
    "http://hgdownload.cse.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes",
    header = None,
    names = ["Chromosome", "End"]
)

chromsizes.insert(1, "Start", 0)
chromsizes.head()

fragments_dict = {'CYHM': '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter_fragments/cyhm_fragments/fragment.tsv.gz',
                'JTHM': '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter_fragments/jthm_fragments/fragment.tsv.gz',
                'QFHM': '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter_fragments/qfhm_fragments/fragment.tsv.gz',
                'MHM': '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter_fragments/mhm_fragments/fragment.tsv.gz',
                'THM': '/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter_fragments/thm_fragments/fragment.tsv.gz'}

In [ ]:
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk
os.makedirs(os.path.join(out_dir, "consensus_peak_calling"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"), exist_ok = True)
os.makedirs(os.path.join(out_dir, 'temp'), exist_ok = True)

bw_paths, bed_paths = export_pseudobulk(
    input_data = cell_data,
    variable = "leiden",
    sample_id_col = "sample",
    chromsizes = chromsizes,
    bed_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"),
    bigwig_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"),
    path_to_fragments = fragments_dict,
    n_cpu = 10,
    normalize_bigwig = True,
    temp_dir = os.path.join(out_dir, 'temp'),
    split_pattern = '-'
)

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv"), "wt") as f:
    for v in bw_paths:
        _ = f.write(f"{v}\t{bw_paths[v]}\n")

with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv"), "wt") as f:
    for v in bed_paths:
        _ = f.write(f"{v}\t{bed_paths[v]}\n")

## infer consensus peaks using masc

In [ ]:
bw_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bw_paths.update({v: p})

bed_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bed_paths.update({v: p})

from pycisTopic.pseudobulk_peak_calling import peak_calling
macs_path = "/home/rsun@ZHANGroup.local/anaconda3/envs/pyscenic/bin/macs2"

os.makedirs(os.path.join(out_dir, "consensus_peak_calling/MACS"), exist_ok = True)

narrow_peak_dict = peak_calling(
    macs_path = macs_path,
    bed_paths = bed_paths,
    outdir = os.path.join(os.path.join(out_dir, "consensus_peak_calling/MACS")),
    genome_size = 'hs',
    n_cpu = 10,
    input_format = 'BEDPE',
    shift = 73,
    ext_size = 146,
    keep_dup = 'all',
    q_value = 0.05,
    _temp_dir = None
)

In [ ]:
from pycisTopic.iterative_peak_calling import get_consensus_peaks
# Other param
peak_half_width=250
path_to_blacklist="/home/rsun@ZHANGroup.local/projects_list/pycisTopic-main/blacklist/mm10-blacklist.v2.bed"
# Get consensus peaks
consensus_peaks = get_consensus_peaks(
    narrow_peaks_dict = narrow_peak_dict,
    peak_half_width = peak_half_width,
    chromsizes = chromsizes,
    path_to_blacklist = path_to_blacklist)

consensus_peaks.to_bed(
    path = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed"),
    keep =True,
    compression = 'infer',
    chain = False)

## QC

In [ ]:
!pycistopic tss gene_annotation_list | grep Mouse
!mkdir -p scenic_noqc_outs/qc
!pycistopic tss get_tss \
    --output scenic_noqc_outs/qc/tss.bed \
    --name "mmusculus_gene_ensembl" \
    --to-chrom-source ucsc \
    --ucsc mm10
!head scenic_noqc_outs/qc/tss.bed | column -t

In [ ]:
## calculate qc metrics parallel 

regions_bed_filename = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed")
tss_bed_filename = os.path.join(out_dir, "qc", "tss.bed")

pycistopic_qc_commands_filename = "pycistopic_qc_commands.txt"

# Create text file with all pycistopic qc command lines.
with open(pycistopic_qc_commands_filename, "w") as fh:
    for sample, fragment_filename in fragments_dict.items():
        print(
            "pycistopic qc",
            f"--fragments {fragment_filename}",
            f"--regions {regions_bed_filename}",
            f"--tss {tss_bed_filename}",
            f"--output {os.path.join(out_dir, 'qc')}/{sample}",
            sep=" ",
            file=fh,
        )

!cat pycistopic_qc_commands.txt | parallel -j 5 {}

### qc plot

In [ ]:
from pycisTopic.plotting.qc_plot import plot_sample_stats, plot_barcode_stats
import matplotlib.pyplot as plt

for sample_id in fragments_dict:
    fig = plot_sample_stats(
        sample_id = sample_id,
        pycistopic_qc_output_dir = "hm_outs/qc"
    )
## 看中间的图的话，这个peak calling的结果是有问题的...

In [ ]:
from pycisTopic.qc import get_barcodes_passing_qc_for_sample
sample_id_to_barcodes_passing_filters = {}
sample_id_to_thresholds = {}

for sample_id in fragments_dict:
    (
        sample_id_to_barcodes_passing_filters[sample_id],
        sample_id_to_thresholds[sample_id]
    ) = get_barcodes_passing_qc_for_sample(
            sample_id = sample_id,
            pycistopic_qc_output_dir = os.path.join(out_dir,'qc'),
            unique_fragments_threshold = 500, # use automatic thresholding
            tss_enrichment_threshold = 0.5, # use automatic thresholding
            frip_threshold = 0,
            use_automatic_thresholds = True,
    )

for sample_id in fragments_dict:
    fig = plot_barcode_stats(
        sample_id = sample_id,
        pycistopic_qc_output_dir = os.path.join(out_dir,'qc'),
        bc_passing_filters = sample_id_to_barcodes_passing_filters[sample_id],
        detailed_title = False,
        **sample_id_to_thresholds[sample_id]
    )

## TSS 分数很低

## create cistopic object

In [ ]:
path_to_regions = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed")
#path_to_blacklist = "pycisTopic/blacklist/hg38-blacklist.v2.bed"
path_to_blacklist="/home/rsun@ZHANGroup.local/projects_list/pycisTopic-main/blacklist/mm10-blacklist.v2.bed"
pycistopic_qc_output_dir = os.path.join(out_dir,'qc')

from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
import polars as pl

cistopic_obj_list = []
for sample_id in fragments_dict:
    sample_metrics = pl.read_parquet(
        os.path.join(pycistopic_qc_output_dir, f'{sample_id}.fragments_stats_per_cb.parquet')
    ).to_pandas().set_index("CB").loc[ sample_id_to_barcodes_passing_filters[sample_id] ]
    cistopic_obj = create_cistopic_object_from_fragments(
        path_to_fragments = fragments_dict[sample_id],
        path_to_regions = path_to_regions,
        path_to_blacklist = path_to_blacklist,
        metrics = sample_metrics,
        valid_bc = sample_id_to_barcodes_passing_filters[sample_id],
        n_cpu = 1,
        project = sample_id,
        split_pattern = '-'
    )
    cistopic_obj_list.append(cistopic_obj)

In [ ]:
for ele in cistopic_obj_list:
    ## 只有很少的细胞能通过QC
    print(ele)

''' 自动化QC的结果
CistopicObject from project CYHM with n_cells × n_regions = 628 × 100050
CistopicObject from project JTHM with n_cells × n_regions = 865 × 119459
CistopicObject from project QFHM with n_cells × n_regions = 399 × 65901
CistopicObject from project MHM with n_cells × n_regions = 431 × 128405
CistopicObject from project THM with n_cells × n_regions = 1161 × 109110
'''

In [ ]:
from pycisTopic.cistopic_class import merge 

cistopic_obj = merge(cistopic_obj_list, split_pattern = '-')
print(cistopic_obj)

import pickle
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

cistopic_obj.add_cell_data(cell_data, split_pattern='-')
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)
pd.crosstab(cistopic_obj.cell_data.loc[:,'species'], cistopic_obj.cell_data.loc[:,'leiden'])

In [ ]:
cistopic_obj.cell_data.head()

## 将cistopic 根据细胞类型进行分组

In [ ]:
cistopic_dir = os.path.join(out_dir, 'leiden_cistopic')
os.makedirs(cistopic_dir, exist_ok = True)

for key in cistopic_obj.cell_data.loc[:,'leiden'].unique():
    sel_idx = cistopic_obj.cell_data.loc[:,'leiden'] == key
    cell_name = cistopic_obj.cell_data.index[sel_idx].values
    for i,x in enumerate(cell_name):
        a,b,c = x.split('-')
        cell_name[i] = '-'.join([a,b])
    cell_name = list(cell_name)
    subobject = cistopic_obj.subset(cell_name, split_pattern = '-', copy = True)
    pickle.dump(
    subobject,
    open(os.path.join(cistopic_dir, f"{key}_cistopic_obj.pkl"), "wb")
)

In [ ]:
import pickle

with open('/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_noqc_outs/leiden_cistopic/0_cistopic_obj.pkl', 'rb') as file:
    obj = pickle.load(file)
obj

## run models

In [ ]:
lda_path = os.path.join(out_dir, 'lda_models')
os.makedirs(lda_path, exist_ok=True)
os.environ['MALLET_MEMORY'] = '200G'
from pycisTopic.lda_models import run_cgs_models_mallet
# Configure path Mallet
mallet_path="/home/rsun@ZHANGroup.local/projects_list/Mallet-202108/bin/mallet"
# Run models
models=run_cgs_models_mallet(
    cistopic_obj,
    n_topics=[2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    n_cpu=128,
    n_iter=500,
    random_state=555,
    alpha=50,
    alpha_by_topic=True,
    eta=0.1,
    eta_by_topic=False,
    tmp_path=lda_path,
    save_path=lda_path,
    mallet_path=mallet_path,
)

In [ ]:
pickle.dump(
    models,
    open(os.path.join(lda_path, "models.pkl"), "wb")
)

## model selection

In [ ]:
from pycisTopic.lda_models import evaluate_models
model = evaluate_models(
    models,
    select_model = 45,
    return_model = True
)

In [ ]:
cistopic_obj.add_LDA_model(model)
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

## cluster and vis

In [ ]:
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

find_clusters(
    cistopic_obj,
    target  = 'cell',
    k = 10,
    res = [1],
    prefix = 'pycisTopic_',
    scale = True,
    split_pattern = '-'
)

run_umap(
    cistopic_obj,
    target  = 'cell', scale=True)

tmp = []
for i,ele in enumerate(cistopic_obj.cell_data.leiden.values):
    tmp.append(str(ele))
cistopic_obj.cell_data.loc[:,'leiden'] = tmp
cistopic_obj.cell_data.leiden

plot_metadata(
    cistopic_obj,
    reduction_name='UMAP',
    variables=['leiden','species'],
    target='cell',
    text_size=10,
    dot_size=5)

In [ ]:
plot_topic(
    cistopic_obj,
    reduction_name = 'UMAP',
    target = 'cell',
    num_columns=5
)

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['leiden'],
    scale = False,
    legend_loc_x = 1.0,
    legend_loc_y = -1.2,
    legend_dist_y = -1,
    figsize = (10, 10)
)

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['species'],
    scale = False,
    legend_loc_x = 1.0,
    legend_loc_y = -1.2,
    legend_dist_y = -1,
    figsize = (10, 10)
)

## topic binarization 

In [ ]:
from pycisTopic.topic_binarization import binarize_topics
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5
)

In [ ]:
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)

In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [ ]:
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img 


topic_qc_metrics = compute_topic_metrics(cistopic_obj)
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

# Plot topic stats in one figure
fig=plt.figure(figsize=(40, 43))
i = 1
for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = fig2img(fig_dict[fig_]) #To convert figures to png to plot together, see .utils.py. This converts the figure to png.
    plt.imshow(img)
    plt.axis('off')
    i += 1
plt.subplots_adjust(wspace=0, hspace=-0.70)
plt.show()

In [ ]:
topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='leiden',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

topic_annot

## DARs

In [ ]:
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

In [ ]:
print(len(variable_regions))

markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='leiden',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.5),
    n_cpu=5,
    _temp_dir=None,
    split_pattern = '-'
)

In [ ]:
from pycisTopic.clust_vis import plot_imputed_features
plot_imputed_features(
    cistopic_obj,
    reduction_name='UMAP',
    imputed_data=imputed_acc_obj,
    features=[markers_dict[x].index.tolist()[0] for x in ['0','5']],
    scale=False,
    num_columns=4
)

print("Number of DARs found:")
print("---------------------")
for x in markers_dict:
    print(f"  {x}: {len(markers_dict[x])}")

## save region sets

In [ ]:
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "DARs_cell_type"), exist_ok = True)
from pycisTopic.utils import region_names_to_coordinates

for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(
        region_bin_topics_top_3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
import pickle
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

In [ ]:
c